# RoadAnomaly — Notebook expérimental (livrable B)

**Projet :** segmentation sémantique binaire supervisée d’anomalies routières  
**Dataset :** RoadAnomaly (Lis et al., ICCV 2019) — *méthode différente* (nous : supervision pixel ; eux : resynthèse)  
**Rapport CRISP-DM associé :** `A_Technical_Report_CRISP_DM.md` (même dossier `deliverable/`)

### Contenu de ce notebook
1. Identité & protocole  
2. Business Understanding  
3. Data Understanding  
4. Data Preparation  
5. Modeling (E0 / E1 / E2a / E2b)  
6. Evaluation (validation + test)  
7. Deployment (Streamlit)  
8. Limites & conclusion  

**Règle :** aucun chiffre inventé ; résultats lus depuis les CSV / métadonnées figés.


## 0. Environnement & chemins

Exécuter depuis le dossier `deliverable/` **ou** la racine du dépôt. Le code détecte automatiquement où se trouvent `experiments/` et `src/`.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE.parent, Path("..").resolve(), Path(".").resolve()]
ROOT = None
for cand in CANDIDATES:
    if (cand / "experiments" / "results" / "phase4").exists() or (
        cand / "supporting_val_comparison.csv"
    ).exists():
        # Prefer repo root if experiments exists; else stay in deliverable
        if (cand / "experiments" / "results" / "phase4").exists():
            ROOT = cand
            break
        ROOT = cand
if ROOT is None:
    ROOT = HERE.parent if (HERE.parent / "src").exists() else HERE

DELIVERABLE = ROOT / "deliverable" if (ROOT / "deliverable").exists() else HERE
SRC = ROOT / "src"
if SRC.exists():
    sys.path.insert(0, str(SRC))

print("ROOT", ROOT)
print("DELIVERABLE", DELIVERABLE)
print("src available", SRC.exists())


## 1. Business Understanding

**Problème :** localiser des obstacles inhabituels sur la chaussée (animaux, cônes, débris, etc.).

**Objectifs :**
- Détecter et **localiser** (masque pixel) l’anomalie  
- Produire une sortie inspectable (image + masque + superposition)  
- Comparer baseline triviale, CNN from scratch, et transfer learning  

**Succès (académique) :** protocole reproductible, métriques Dice/IoU honnêtes, analyse des limites (petit n, déséquilibre).  
**Hors scope :** système ADAS certifié ; reproduction exacte de Lis et al. 2019.


## 2. Data Understanding

- **60** images web annotées (labels sémantiques : fond=0, anomalie=2)  
- ~**9,85 %** de pixels anomalie (fort déséquilibre → l’accuracy seule est trompeuse)  
- **Pas** de sous-ensemble « normal only » → autoencodeur classique non justifié comme piste principale  
- Sources / crédits : `credits.txt` à la racine du dépôt  

Exploration détaillée (figures) : `notebooks/01_exploration_and_preprocessing.ipynb`.


## 3. Data Preparation

Pipeline figé (`src/preprocessing.py`, `dataset.py`, `augmentation.py`) :

| Étape | Choix |
| --- | --- |
| Resize | **512×288** (W×H), sans letterbox |
| Labels | {0,2} → {0,1} |
| Split | **42 / 9 / 9** (train / val / test), seed 42 |
| Augmentations | train only |
| Normalisation E1 | `scale_0_1` |
| Normalisation E2 | `mobilenet_v2` (≈[-1, 1]) |

Verrou de split (SHA-256) :  
`9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78`


In [ ]:
EXPECTED_SPLIT = "9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78"
try:
    from dataset import load_split_manifest, verify_split_lock
    manifest = load_split_manifest()
    print(manifest["split"].value_counts().to_dict())
    sha = verify_split_lock(manifest)
    print("split_sha", sha)
    print("lock_ok", sha == EXPECTED_SPLIT)
except Exception as exc:
    print("Split check skipped (src/data not available in this environment):", type(exc).__name__, exc)
    print("Expected SHA:", EXPECTED_SPLIT)


## 4. Modeling

### E0 — Baseline « tout fond »
Tous les pixels prédits fond. Illustre le piège de l’accuracy.

### E1 — CNN U-Net-lite from scratch
- `src/models/cnn.py` — `base_filters=8`, **sans BatchNorm**  
- Loss : BCE pondérée (pos=15) + soft Dice  
- Adam 1e-3, batch 2, seed 42  
- Checkpoint : max `val_mean_dice` @ seuil train **0,5**  
- Run canonique : `E1_cnn_scratch_20261108_1510`

### E2a — U-Net + MobileNetV2 (encodeur gelé)
- `src/models/unet_mobilenetv2.py`, `configs/phase4.json`  
- Encodeur ImageNet gelé, décodeur entraîné  
- Normalisation `mobilenet_v2`

### E2b — Fine-tuning sélectif (modèle primaire)
- Initialise depuis E2a ; dégel à partir de `block_13_expand`  
- LR plus faible (1e-4)  
- Seuil opérationnel choisi sur **validation** : **0,55**  
- Expérience : `E2b_unet_mobilenetv2_finetune_20261108_1721`

Entraînement Colab (déjà exécuté) : `notebooks/02_phase4_training_colab.ipynb`.


In [ ]:
# Aperçu architecture E1 / E2 (construction sans poids ImageNet pour E2)
try:
    from models.cnn import build_scratch_fcn
    from models.unet_mobilenetv2 import build_unet_mobilenetv2
    e1 = build_scratch_fcn(288, 512, base_filters=8, dropout=0.2)
    e2 = build_unet_mobilenetv2(input_height=288, input_width=512, weights=None, encoder_trainable=False)
    print("E1:", e1.name, "params~", e1.count_params())
    print("E2:", e2.name, "params~", e2.count_params())
except Exception as exc:
    print("Model build skipped:", type(exc).__name__, exc)


## 5. Evaluation — Validation (sélection de modèle)

Métrique de sélection : **Dice moyen par image** sur val, après grille de seuils.  
**Modèle primaire figé :** E2b @ seuil **0,55** (`FINAL_MODEL_SELECTION_FREEZE.json`).


In [ ]:
val_paths = [
    DELIVERABLE / "supporting_val_comparison.csv",
    ROOT / "experiments/results/phase4/phase4_E0_E1_E2_val_comparison.csv",
]
val_csv = next((p for p in val_paths if p.is_file()), None)
assert val_csv is not None, "Val comparison CSV missing"
val = pd.read_csv(val_csv)
cols = [c for c in ["experiment", "experiment_id", "threshold", "mean_dice", "mean_iou", "mean_precision", "mean_recall", "mean_accuracy"] if c in val.columns]
print("Source:", val_csv)
display(val[cols]) if "display" in dir() else print(val[cols].to_string(index=False))

meta = ROOT / "experiments/metadata/phase4/E2b_unet_mobilenetv2_finetune_20261108_1721.json"
if meta.is_file():
    e2b = json.loads(meta.read_text(encoding="utf-8"))
    print("E2b operational thr", e2b.get("threshold_selection", {}).get("value"))
    print("E2b status", e2b.get("status"))


## 6. Evaluation — Test (one-shot, après gel)

Règles : **pas** de retuning de seuil ; **pas** de re-sélection de modèle ; une seule passe.  
Notebook d’exécution : `notebooks/03_final_test_evaluation_colab.ipynb`.


In [ ]:
test_paths = [
    DELIVERABLE / "supporting_test_comparison.csv",
    ROOT / "experiments/results/phase4/phase4_final_test_comparison.csv",
]
test_csv = next((p for p in test_paths if p.is_file()), None)
assert test_csv is not None, "Test comparison CSV missing — do not invent metrics"
test = pd.read_csv(test_csv)
keep = [c for c in ["experiment_id", "threshold", "mean_dice", "mean_iou", "mean_precision", "mean_recall", "global_dice", "global_accuracy"] if c in test.columns]
print("Source:", test_csv)
display(test[keep]) if "display" in dir() else print(test[keep].to_string(index=False))
print("Primary on test: E2b mean_dice ≈", float(test.loc[test["experiment_id"].str.startswith("E2b"), "mean_dice"].iloc[0]))


## 7. Deployment (démonstration)

Application Streamlit française (`app.py` à la racine du dépôt) :

```powershell
pip install -r requirements-app.txt
streamlit run app.py
```

- Inférence **officielle** : E2b, seuil figé **0,55**  
- Onglet **Exploration visuelle** : seuil d’affichage hors protocole (ne retune pas le modèle)  
- Guide : `docs/APP_USER_GUIDE.md`


## 8. Limites & conclusion

**Limites :** n_train=42, n_val=n_test=9 ; déséquilibre de classes ; variance élevée ; segmentation parfois partielle sur gros obstacles ; pas de lost-tire en test.

**Conclusion :** le transfer learning MobileNetV2 (E2b) reste le meilleur modèle en validation **et** sur le test one-shot (Dice mean/image ≈ 0,818). Les conclusions restent conditionnées à un jeu minuscule.

**Rapport complet :** `A_Technical_Report_CRISP_DM.md`.
